# Teste da função build_modeling_dataset

Este notebook valida a lógica da função responsável pela construção
do dataset de modelagem antes da utilização dos dados reais.

In [1]:
import sys
from pathlib import Path

print("Diretório atual:")
print(Path.cwd())

print("\nDiretório pai:")
print(Path.cwd().parent)

Diretório atual:
c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\notebooks

Diretório pai:
c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.build_modeling_dataset import build_modeling_dataset

In [3]:
print(PROJECT_ROOT)

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3


criar dados artificiais de alunos

In [8]:
import pandas as pd


alunos_teste = pd.DataFrame(
    {
        "ano": [2024, 2024, 2024],
        "id_aluno": ["A1", "A2", "A4"],
        "id_municipio": ["M1", "M1", "M3"],
        "id_escola": ["E1", "E1", "E3"],
        "rede": ["2", "2", "3"],
        "alfabetizado": ["1", "0", "Sim"],
        "peso_aluno": [1.0, 1.2, 0.8],
    }
)

alunos_teste

,ano,id_aluno,id_municipio,id_escola,rede,alfabetizado,peso_aluno
0,2024,A1,M1,E1,2,1,1.0
1,2024,A2,M1,E1,2,0,1.2
2,2024,A4,M3,E3,3,Sim,0.8


criar histórico municipal artificial

In [9]:
municipio_teste = pd.DataFrame(
    {
        "ano": [2023, 2023],
        "id_municipio": ["M1", "M2"],
        "rede": ["2", "3"],
        "taxa_alfabetizacao": [62.5, 55.0],
        "media_portugues": [750.0, 720.0],
    }
)

municipio_teste

,ano,id_municipio,rede,taxa_alfabetizacao,media_portugues
0,2023,M1,2,62.5,750.0
1,2023,M2,3,55.0,720.0


executar a função

In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.build_modeling_dataset import build_modeling_dataset

print("Função importada com sucesso.")

Função importada com sucesso.


In [15]:
dataset_teste = build_modeling_dataset(
    alunos_df=alunos_teste,
    municipio_df=municipio_teste,
)

dataset_teste

,ano,id_aluno,id_municipio,id_escola,rede,peso_aluno,alfabetizado,taxa_alfabetizacao_2023,media_portugues_2023,historico_2023_disponivel
0,2024,A1,M1,E1,2,1.0,1,62.5,750.0,1
1,2024,A2,M1,E1,2,1.2,0,62.5,750.0,1
2,2024,A4,M3,E3,3,0.8,1,NaN,NaN,0


conferir manualmente

In [16]:
dataset_teste[
    [
        "id_aluno",
        "rede",
        "alfabetizado",
        "taxa_alfabetizacao_2023",
        "media_portugues_2023",
        "historico_2023_disponivel",
    ]
]

,id_aluno,rede,alfabetizado,taxa_alfabetizacao_2023,media_portugues_2023,historico_2023_disponivel
0,A1,2,1,62.5,750.0,1
1,A2,2,0,62.5,750.0,1
2,A4,3,1,NaN,NaN,0


validar automaticamente

In [17]:
assert dataset_teste.shape[0] == 3

assert set(dataset_teste["id_aluno"]) == {
    "A1",
    "A2",
    "A4",
}

assert dataset_teste["id_aluno"].is_unique

assert dataset_teste["alfabetizado"].tolist() == [
    1,
    0,
    1,
]

assert (
    dataset_teste.loc[
        dataset_teste["id_aluno"] == "A4",
        "historico_2023_disponivel",
    ].iloc[0]
    == 0
)

assert (
    dataset_teste.loc[
        dataset_teste["id_aluno"] == "A4",
        "taxa_alfabetizacao_2023",
    ].isna().iloc[0]
)

assert (
    dataset_teste.loc[
        dataset_teste["id_aluno"] == "A1",
        "media_portugues_2023",
    ].iloc[0]
    == 750.0
)

print("Todos os testes passaram.")

Todos os testes passaram.


Teste mínimo com BigQuery

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.bigquery_reader import run_query

teste = run_query(
    query="SELECT 1 AS teste",
    project_id="projeto-fiap-grupo-x",
)

teste

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,teste
0,1


In [2]:
from src.preprocessing.bigquery_reader import run_query

teste_bigquery = run_query(
    query="SELECT 1 AS teste",
    project_id="projeto-fiap-grupo-x",
)

teste_bigquery

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,teste
0,1


In [3]:
from src.preprocessing.queries import (
    QUERY_ALUNOS_2024,
    QUERY_MUNICIPIO_2023,
)

In [4]:
alunos_amostra = run_query(
    query=QUERY_ALUNOS_2024 + "\nLIMIT 10",
    project_id="projeto-fiap-grupo-x",
)

alunos_amostra

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ano,id_municipio,id_escola,id_aluno,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2024,1100205,60000343,11022554,3,0,0,0,NaN,NaN
1,2024,1303536,60000944,13020494,3,0,0,0,NaN,NaN
2,2024,1506807,60001681,15073828,3,0,0,0,NaN,NaN
3,2024,1709500,60004023,17018735,3,0,0,0,NaN,NaN
4,2024,2105302,60005799,21067439,3,0,0,0,NaN,NaN
5,2024,2206670,60007034,22031723,3,0,0,0,NaN,NaN
6,2024,2403400,60009787,24027409,3,0,0,0,NaN,NaN
7,2024,2607208,60013070,26078623,3,0,0,0,NaN,NaN
8,2024,2607901,60012336,26038465,3,0,0,0,NaN,NaN
9,2024,2611606,60012963,26054269,3,0,0,0,NaN,NaN


In [5]:
municipio_amostra = run_query(
    query=QUERY_MUNICIPIO_2023 + "\nLIMIT 10",
    project_id="projeto-fiap-grupo-x",
)

municipio_amostra

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ano,id_municipio,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1101609,3,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1101807,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023,1302900,3,53.17,731.0744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023,1303304,2,72.32,755.2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023,1500107,5,39.73,725.5913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023,1501105,5,46.70,733.3467,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2023,1501725,3,73.21,758.3553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


validar schema e cobertura antes da extração completa

In [6]:
print("ALUNOS")
print(alunos_amostra.dtypes)

print("\nMUNICÍPIO")
print(municipio_amostra.dtypes)

ALUNOS
ano                        Int64
id_municipio                 str
id_escola                    str
id_aluno                     str
rede                         str
presenca                     str
preenchimento_caderno        str
alfabetizado                 str
proficiencia             float64
peso_aluno               float64
dtype: object

MUNICÍPIO
ano                          Int64
id_municipio                   str
rede                           str
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    float64
proporcao_aluno_nivel_3    float64
proporcao_aluno_nivel_4    float64
proporcao_aluno_nivel_5    float64
proporcao_aluno_nivel_6    float64
proporcao_aluno_nivel_7    float64
proporcao_aluno_nivel_8    float64
dtype: object


In [7]:
query_validacao_municipio = """
SELECT
    ano,
    rede,
    COUNT(*) AS total_registros,

    COUNTIF(taxa_alfabetizacao IS NULL)
        AS taxa_alfabetizacao_nula,

    COUNTIF(media_portugues IS NULL)
        AS media_portugues_nula,

    COUNTIF(proporcao_aluno_nivel_0 IS NULL)
        AS nivel_0_nulo,

    COUNTIF(proporcao_aluno_nivel_1 IS NULL)
        AS nivel_1_nulo,

    COUNTIF(proporcao_aluno_nivel_2 IS NULL)
        AS nivel_2_nulo,

    COUNTIF(proporcao_aluno_nivel_3 IS NULL)
        AS nivel_3_nulo,

    COUNTIF(proporcao_aluno_nivel_4 IS NULL)
        AS nivel_4_nulo,

    COUNTIF(proporcao_aluno_nivel_5 IS NULL)
        AS nivel_5_nulo,

    COUNTIF(proporcao_aluno_nivel_6 IS NULL)
        AS nivel_6_nulo,

    COUNTIF(proporcao_aluno_nivel_7 IS NULL)
        AS nivel_7_nulo,

    COUNTIF(proporcao_aluno_nivel_8 IS NULL)
        AS nivel_8_nulo

FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio`

WHERE ano = 2023

GROUP BY
    ano,
    rede

ORDER BY
    rede
"""

validacao_municipio = run_query(
    query=query_validacao_municipio,
    project_id="projeto-fiap-grupo-x",
)

validacao_municipio

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ano,rede,total_registros,taxa_alfabetizacao_nula,media_portugues_nula,nivel_0_nulo,nivel_1_nulo,nivel_2_nulo,nivel_3_nulo,nivel_4_nulo,nivel_5_nulo,nivel_6_nulo,nivel_7_nulo,nivel_8_nulo
0,2023,2,1149,0,0,1149,1149,1149,1149,1149,1149,1149,1149,1149
1,2023,3,5448,0,0,5448,5448,5448,5448,5448,5448,5448,5448,5448
2,2023,5,4950,0,0,4950,4950,4950,4950,4950,4950,4950,4950,4950


Extração completa

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.preprocessing.queries import (
    QUERY_ALUNOS_2024,
    QUERY_MUNICIPIO_2023,
)

from src.preprocessing.bigquery_reader import run_query

from src.preprocessing.build_modeling_dataset import (
    build_modeling_dataset,
)

In [3]:
print(QUERY_ALUNOS_2024)
print(QUERY_MUNICIPIO_2023)


SELECT
    ano,
    id_municipio,
    id_escola,
    id_aluno,
    rede,
    alfabetizado,
    peso_aluno
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
WHERE
    ano = 2024
    AND presenca = '1'
    AND preenchimento_caderno = '1'


SELECT
    ano,
    id_municipio,
    rede,
    taxa_alfabetizacao,
    media_portugues
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio`
WHERE ano = 2023



In [4]:
from src.preprocessing.queries import (
    QUERY_ALUNOS_2024,
    QUERY_MUNICIPIO_2023,
)

alunos_2024 = run_query(
    query=QUERY_ALUNOS_2024,
    project_id="projeto-fiap-grupo-x",
)

municipio_2023 = run_query(
    query=QUERY_MUNICIPIO_2023,
    project_id="projeto-fiap-grupo-x",
)

print("Alunos 2024:", alunos_2024.shape)
print("Município 2023:", municipio_2023.shape)

c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Alunos 2024: (1851852, 7)
Município 2023: (11547, 5)


Construir modeling_dataset_2024

In [5]:
modeling_dataset_2024 = build_modeling_dataset(
    alunos_df=alunos_2024,
    municipio_df=municipio_2023,
)

print("Dataset construído.")
print("Shape:", modeling_dataset_2024.shape)

Dataset construído.
Shape: (1851852, 10)


In [6]:
print("=== VALIDAÇÃO DO DATASET DE MODELAGEM ===")

print("\nShape:")
print(modeling_dataset_2024.shape)

print("\nAlunos duplicados:")
print(modeling_dataset_2024["id_aluno"].duplicated().sum())

print("\nDistribuição do target:")
print(
    modeling_dataset_2024["alfabetizado"]
    .value_counts()
    .sort_index()
)

print("\nDistribuição percentual do target:")
print(
    modeling_dataset_2024["alfabetizado"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nDisponibilidade do histórico 2023:")
print(
    modeling_dataset_2024[
        "historico_2023_disponivel"
    ].value_counts().sort_index()
)

print("\nValores nulos:")
print(
    modeling_dataset_2024
    .isna()
    .sum()
    .sort_values(ascending=False)
)

=== VALIDAÇÃO DO DATASET DE MODELAGEM ===

Shape:
(1851852, 10)

Alunos duplicados:
0

Distribuição do target:
alfabetizado
0     744733
1    1107119
Name: count, dtype: int64

Distribuição percentual do target:
alfabetizado
0    40.22
1    59.78
Name: proportion, dtype: float64

Disponibilidade do histórico 2023:
historico_2023_disponivel
0      35582
1    1816270
Name: count, dtype: int64

Valores nulos:
taxa_alfabetizacao_2023      35582
media_portugues_2023         35582
id_aluno                         0
ano                              0
id_municipio                     0
id_escola                        0
peso_aluno                       0
rede                             0
alfabetizado                     0
historico_2023_disponivel        0
dtype: int64


validação final antes de salvar

In [7]:
# Validações finais do dataset de modelagem

assert modeling_dataset_2024.shape[0] == 1_851_852, \
    "Quantidade inesperada de registros."

assert modeling_dataset_2024["id_aluno"].is_unique, \
    "Existem alunos duplicados."

assert modeling_dataset_2024["alfabetizado"].isna().sum() == 0, \
    "Existem targets nulos."

assert set(modeling_dataset_2024["alfabetizado"].unique()) == {0, 1}, \
    "Target contém valores diferentes de 0 e 1."

assert modeling_dataset_2024["id_municipio"].isna().sum() == 0, \
    "Existem municípios nulos."

assert modeling_dataset_2024["rede"].isna().sum() == 0, \
    "Existem redes nulas."

assert modeling_dataset_2024["peso_aluno"].isna().sum() == 0, \
    "Existem pesos amostrais nulos."

assert (
    modeling_dataset_2024["historico_2023_disponivel"].sum()
    == 1_816_270
), "Quantidade inesperada de alunos com histórico."

assert (
    modeling_dataset_2024["taxa_alfabetizacao_2023"].isna()
    ==
    modeling_dataset_2024["media_portugues_2023"].isna()
).all(), "Features históricas apresentam padrões de ausência diferentes."

assert (
    modeling_dataset_2024["historico_2023_disponivel"]
    ==
    modeling_dataset_2024["taxa_alfabetizacao_2023"].notna().astype("int8")
).all(), "Flag de histórico inconsistente."

print("✅ Todas as validações finais passaram.")

✅ Todas as validações finais passaram.


In [8]:
print(modeling_dataset_2024.columns.tolist())

['ano', 'id_aluno', 'id_municipio', 'id_escola', 'rede', 'peso_aluno', 'alfabetizado', 'taxa_alfabetizacao_2023', 'media_portugues_2023', 'historico_2023_disponivel']


Salvar o dataset de modelagem

In [9]:
from pathlib import Path

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "modeling_dataset_2024.parquet"

modeling_dataset_2024.to_parquet(
    output_path,
    index=False,
)

print(f"Dataset salvo em: {output_path}")

Dataset salvo em: c:\Users\andre\OneDrive\Documentos\POSTECH - IA Scientist\tech-challenge-fase3\data\processed\modeling_dataset_2024.parquet


In [10]:
import pandas as pd

dataset_recarregado = pd.read_parquet(output_path)

print("=== VALIDAÇÃO DO PARQUET ===")
print("Shape:", dataset_recarregado.shape)
print("Duplicidades:", dataset_recarregado["id_aluno"].duplicated().sum())
print("Target nulo:", dataset_recarregado["alfabetizado"].isna().sum())
print("Colunas:", dataset_recarregado.columns.tolist())

=== VALIDAÇÃO DO PARQUET ===
Shape: (1851852, 10)
Duplicidades: 0
Target nulo: 0
Colunas: ['ano', 'id_aluno', 'id_municipio', 'id_escola', 'rede', 'peso_aluno', 'alfabetizado', 'taxa_alfabetizacao_2023', 'media_portugues_2023', 'historico_2023_disponivel']


In [11]:
tamanho_mb = output_path.stat().st_size / (1024 ** 2)

print(f"Tamanho do arquivo: {tamanho_mb:.2f} MB")

Tamanho do arquivo: 26.07 MB
